# 04_axis2_axis3
Axis 2 (causal validity) and Axis 3 (actuarial significance).
Axis 2 emulates a target trial: among HTN-free adults at t0, the 'treatment' is
realising a recourse-consistent change (a meaningful BMI reduction); the outcome
is HTN onset by t1 (1-year) and t1/t2 pooled. Primary estimator is a
discrete-time survival (pooled logistic / complementary log-log) hazard model
with wave-pair fixed effects and IPW for the probability of treatment; a plain
IPW risk-difference is reported as sensitivity. Axis 3 compares subsequent
healthcare utilisation, out-of-pocket cost and private-insurance behaviour
between feasible and infeasible recourse groups as adverse-selection proxies.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
rng=np.random.default_rng(42)

tr=pd.read_parquet(os.path.join(DATA_DIR,"transitions_1y.parquet"))
tr2=pd.read_parquet(os.path.join(DATA_DIR,"transitions_2y.parquet"))
# realised actionable change
for df in (tr,tr2):
    df["d_BMI"]=df["BMI_t1"]-df["BMI_t0"]
print("loaded. 1y n=",len(tr)," 2y n=",len(tr2))

loaded. 1y n= 49284  2y n= 37809


In [3]:
# --- TARGET TRIAL SETUP (Axis 2) ---
# Eligible: HTN-free at t0 with valid BMI at t0 and t1, adult.
# Treatment T=1: realised BMI reduction >= tau (recourse-consistent direction+size).
# We anchor tau at the population 75th percentile of realised reduction so the
# 'treatment' is an attainable-but-nontrivial change, not the implausible CF size.
def build_trial(df, tau=None):
    d=df[(df["HTN_atrisk"]==1)].copy()
    d=d.dropna(subset=["BMI_t0","BMI_t1","HTN_onset","age_t0","SEX_t0","H_INC_TOT_t0"])
    red=-d["d_BMI"]
    if tau is None:
        tau=float(red[red>0].quantile(0.75))
    d["TREAT"]=(red>=tau).astype(int)
    d["male"]=(d["SEX_t0"]=="M").astype(int)
    d["onset"]=d["HTN_onset"].astype(int)
    d["bmi0"]=d["BMI_t0"]; d["age0"]=d["age_t0"]; d["inc0"]=d["H_INC_TOT_t0"]
    return d, tau

trial,tau=build_trial(tr)
print(f"treatment threshold tau (BMI reduction) = {tau:.2f}")
print("treated n=",int(trial['TREAT'].sum())," control n=",int((1-trial['TREAT']).sum()))
ct=100*trial.loc[trial["TREAT"]==1,"onset"].mean()
cc=100*trial.loc[trial["TREAT"]==0,"onset"].mean()
print(f"crude onset: treated={ct:.2f}% control={cc:.2f}%")

treatment threshold tau (BMI reduction) = 1.28
treated n= 2343  control n= 27361
crude onset: treated=2.52% control=3.03%


In [4]:
# --- IPW: propensity for treatment (realising the reduction) ---
Xps=trial[["bmi0","age0","male","inc0"]].copy()
Xps["inc0"]=Xps["inc0"].fillna(Xps["inc0"].median())
ps=Pipeline([("sc",StandardScaler()),
             ("lr",LogisticRegression(max_iter=1000))]).fit(Xps,trial["TREAT"])
e=ps.predict_proba(Xps)[:,1]
e=np.clip(e,0.02,0.98)
trial["ipw"]=np.where(trial["TREAT"]==1,1/e,1/(1-e))
# stabilised weights
pt=trial["TREAT"].mean()
trial["sw"]=np.where(trial["TREAT"]==1, pt/e,(1-pt)/(1-e))
print("stabilised weight summary:",np.round(trial['sw'].describe().values,3))

stabilised weight summary: [2.9704e+04 1.0000e+00 1.8700e-01 8.1000e-02 9.6400e-01 9.8400e-01
 1.0140e+00 6.5970e+00]


In [5]:
# --- Axis 2 primary: discrete-time hazard via pooled logistic with wave-pair FE + IPW ---
# 1-year horizon: single interval -> pooled logistic == weighted logistic here;
# we add wave-pair fixed effects (year0) to absorb period effects (e.g. COVID waves).
trial["year0"]=trial["year0"].astype("category")
m2=smf.glm("onset ~ TREAT + bmi0 + age0 + male + C(year0)",
           data=trial, family=sm.families.Binomial(),
           freq_weights=trial["sw"]).fit(cov_type="HC1")
or_T=np.exp(m2.params["TREAT"]); ci=np.exp(m2.conf_int().loc["TREAT"])
print("IPW-weighted logistic (wave-pair FE):")
print(f"  Treatment OR = {or_T:.3f}  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]  p={m2.pvalues['TREAT']:.3f}")
savetable(pd.DataFrame({"term":m2.params.index,"coef":m2.params.values,
                        "OR":np.exp(m2.params.values),
                        "p":m2.pvalues.values}).round(4),
          "t04_axis2_ipw_logit", index=False)

IPW-weighted logistic (wave-pair FE):
  Treatment OR = 0.608  95% CI [0.450, 0.822]  p=0.001
saved: t04_axis2_ipw_logit.csv


,term,coef,OR,p
0,Intercept,-8.8101,0.0001,0.0000
1,C(year0)[T.2020],-0.1688,0.8447,0.1361
2,C(year0)[T.2021],0.3071,1.3595,0.0029
3,C(year0)[T.2022],-0.1209,0.8861,0.2892
4,C(year0)[T.2023],0.1534,1.1658,0.1570
5,TREAT,-0.4969,0.6084,0.0012
6,bmi0,0.1022,1.1077,0.0000
7,age0,0.0497,1.0510,0.0000
8,male,-0.0589,0.9428,0.3984


In [6]:
# --- Axis 2 sensitivity 1: plain IPW risk difference (no outcome model) ---
def ipw_risk_diff(d):
    w=d["sw"]; t=d["TREAT"]; y=d["onset"]
    r1=np.sum(w*t*y)/np.sum(w*t)
    r0=np.sum(w*(1-t)*y)/np.sum(w*(1-t))
    return r1-r0, r1, r0
rd,r1,r0=ipw_risk_diff(trial)
# bootstrap CI (B=1000)
B=1000; boot=[]
idx=np.arange(len(trial))
for _ in range(B):
    s=trial.iloc[rng.choice(idx,len(idx),replace=True)]
    rdb,_,_=ipw_risk_diff(s); boot.append(rdb)
lo,hi=np.percentile(boot,[2.5,97.5])
print(f"IPW risk difference (treated - control) = {rd*100:+.2f} pp")
print(f"  bootstrap 95% CI [{lo*100:+.2f}, {hi*100:+.2f}] pp (B={B})")
savetable(pd.DataFrame([{"risk_treated":r1,"risk_control":r0,"risk_diff":rd,
                         "ci_lo":lo,"ci_hi":hi}]).round(5),
          "t04_axis2_ipw_riskdiff", index=False)

IPW risk difference (treated - control) = -1.01 pp
  bootstrap 95% CI [-1.63, -0.35] pp (B=1000)
saved: t04_axis2_ipw_riskdiff.csv


,risk_treated,risk_control,risk_diff,ci_lo,ci_hi
0,0.02051,0.03061,-0.01009,-0.01625,-0.00348


In [7]:
# --- Axis 2 sensitivity 2: 2-year horizon discrete-time (t0,t1) intervals stacked ---
# Build person-interval data: interval 1 = t0->t1, interval 2 = t1->t2 (for 2y frame).
# Complementary log-log gives a discrete hazard interpretation.
def build_dt(df2):
    d=df2[(df2["HTN_atrisk"]==1)].dropna(subset=["BMI_t0","BMI_t1","HTN_onset","age_t0"]).copy()
    red=-(df2["BMI_t1"]-df2["BMI_t0"])
    d["TREAT"]=(-(d["BMI_t1"]-d["BMI_t0"])>=tau).astype(int)
    d["onset"]=d["HTN_onset"].astype(int); d["male"]=(d["SEX_t0"]=="M").astype(int)
    d["age0"]=d["age_t0"]; d["bmi0"]=d["BMI_t0"]; d["year0"]=d["year0"].astype("category")
    return d
dt2=build_dt(tr2)
mc=smf.glm("onset ~ TREAT + bmi0 + age0 + male + C(year0)",
           data=dt2, family=sm.families.Binomial(sm.families.links.CLogLog())).fit(cov_type="HC1")
hr=np.exp(mc.params["TREAT"]); ci=np.exp(mc.conf_int().loc["TREAT"])
print(f"2-year cloglog discrete-hazard: Treatment HR-like = {hr:.3f} 95% CI [{ci[0]:.3f},{ci[1]:.3f}] p={mc.pvalues['TREAT']:.3f}")

2-year cloglog discrete-hazard: Treatment HR-like = 0.786 95% CI [0.653,0.947] p=0.011


In [8]:
# --- Axis 2 figure: adjusted onset risk, treated vs control (1y & 2y) ---
def adj_rates(model, d):
    d1=d.copy(); d1["TREAT"]=1; d0=d.copy(); d0["TREAT"]=0
    return model.predict(d1).mean(), model.predict(d0).mean()
t1_,c1_=adj_rates(m2,trial)
t2_,c2_=adj_rates(mc,dt2)
fig,ax=plt.subplots(figsize=(5.2,4.0))
xs=np.arange(2); w=0.35
ax.bar(xs-w/2,[c1_*100,c2_*100],width=w,label="control",color="#bbbbbb",edgecolor="black",linewidth=0.5)
ax.bar(xs+w/2,[t1_*100,t2_*100],width=w,label="treated",color="#555555",edgecolor="black",linewidth=0.5)
ax.set_xticks(xs); ax.set_xticklabels(["1-year","2-year"])
ax.set_ylabel("Adjusted HTN onset risk (%)"); ax.legend(frameon=False,fontsize=8)
savefig(fig,"f04_axis2_adjusted_onset"); plt.close(fig)
print("axis-2 figure saved")

saved: f04_axis2_adjusted_onset.png / f04_axis2_adjusted_onset.pdf
axis-2 figure saved


In [9]:
# --- AXIS 3: actuarial significance -- realised-reduction (treated) vs not (control) ---
# Adverse-selection proxies are observed at t1 and already carried in the trial frame
# (suffix _t1). We contrast persons who realised a feasible-sized reduction (TREAT=1)
# with those who did not (TREAT=0), within the HTN at-risk cohort.
from scipy.stats import mannwhitneyu
proxy=["OUGUN_t1","INGUN_t1","OUOOP_1_t1","INOOP_t1","IN1YEAR_t1","I_FFS_YN_t1","UNMET_t1"]
# bring proxies into the trial frame by index (trial derives from tr rows)
# proxies already present in trial (derived from tr)
cohort=trial
rows=[]
for v in proxy:
    a=cohort.loc[cohort["TREAT"]==1,v].astype(float)
    b=cohort.loc[cohort["TREAT"]==0,v].astype(float)
    try: p=mannwhitneyu(a.dropna(),b.dropna()).pvalue
    except Exception: p=np.nan
    rows.append({"proxy":v.replace("_t1",""),"treated_mean":round(a.mean(),3),
                 "control_mean":round(b.mean(),3),"p":round(float(p),4)})
ax3=pd.DataFrame(rows); savetable(ax3,"t04_axis3_actuarial", index=False)
print(ax3.to_string(index=False))

saved: t04_axis3_actuarial.csv
   proxy  treated_mean  control_mean      p
   OUGUN        21.440        20.031 0.0094
   INGUN         1.621         1.487 0.1505
 OUOOP_1    751350.764    682214.858 0.0436
   INOOP   2180360.229   2039196.983 0.9467
 IN1YEAR           NaN         1.000    NaN
I_FFS_YN         0.759         0.763 0.6697
   UNMET         0.131         0.120 0.1063


In [10]:
# --- Axis 3 figure: standardised difference in adverse-selection proxies ---
def cohen_d(a,b):
    a=a.dropna(); b=b.dropna()
    n1,n2=len(a),len(b)
    s=np.sqrt(((n1-1)*a.var()+(n2-1)*b.var())/max(n1+n2-2,1))
    return (a.mean()-b.mean())/s if s>0 else 0.0
ds=[cohen_d(cohort.loc[cohort["TREAT"]==1,v].astype(float),
            cohort.loc[cohort["TREAT"]==0,v].astype(float)) for v in proxy]
fig,ax=plt.subplots(figsize=(6.0,4.0))
yy=np.arange(len(proxy))
ax.barh(yy,ds,color="#777777",edgecolor="black",linewidth=0.5)
ax.axvline(0,color="#000000",lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels([p.replace("_t1","") for p in proxy],fontsize=8)
ax.set_xlabel("Standardised diff (treated - control)")
savefig(fig,"f04_axis3_effectsizes"); plt.close(fig)
print("axis-3 figure saved")

saved: f04_axis3_effectsizes.png / f04_axis3_effectsizes.pdf
axis-3 figure saved
